# 08. Mitigation and abstention (E10, E11)

- Data: D1, outdoor vs. indoor (leak rows, 34,663)
- Protocol: site-independent split (GroupShuffleSplit by site); test sites 30% (E10), 40% (E11)
- 5 seeds; training rows capped at 20,000
- Majority rate 0.560: mean over seeds of the test-site majority rate (outdoor in 4 seeds, indoor in seed 2); pooled rate 0.569

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation as ev, sfb, reporting as rp
cfg.apply_mpl_style(); cfg.banner()
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import warnings; warnings.filterwarnings("ignore")

X1, X1_max, M1, HZ = io_utils.load_d1()
y5 = M1.cls.to_numpy(); site = M1.site.to_numpy()
mask = np.isin(y5, ["out", "in"])
Xoi, yoi, soi = X1[mask], y5[mask], site[mask]
CAP = cfg.D1_SUBSAMPLE_MITIGATION
SEEDS = range(cfg.N_SEEDS_AUX)
print(f"D1 outdoor vs indoor: {len(yoi):,} rows, {(yoi=='out').sum()} out / {(yoi=='in').sum()} in, "
      f"{len(set(soi))} sites; pooled majority = {sfb.majority_baseline(yoi):.4f}")

def site_split(seed, test_size):
    return next(GroupShuffleSplit(1, test_size=test_size, random_state=seed)
                .split(np.arange(len(yoi)), yoi, groups=soi))

base_rows = []
for seed in SEEDS:
    a, b = site_split(seed, .3)
    base_rows.append(dict(seed=seed, n_test_sites=len(set(soi[b])), n_test=len(b),
                          test_outdoor_share=float(np.mean(yoi[b] == "out"))))
bt = pd.DataFrame(base_rows)
print(bt.to_string(index=False))
print(f"mean test-fold majority (outdoor share) = {bt.test_outdoor_share.mean():.4f}  <- the 0.560 of v1")
la.save_json(dict(benchmark="D1", task="TR_out_vs_in", protocol="site-independent (GroupShuffleSplit by site)",
                  test_size_E10=0.3, test_size_E11=0.4, seeds=list(SEEDS), train_cap=CAP,
                  pooled_majority=sfb.majority_baseline(yoi),
                  mean_test_majority_E10=float(bt.test_outdoor_share.mean())), "E10_E11_setting.json")
rp.record("D1.E10.majority_mean_testfold", bt.test_outdoor_share.mean(), "E10_E11_setting.json", "R1-M9")

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000
D1 outdoor vs indoor: 34,663 rows, 19731 out / 14932 in, 48 sites; pooled majority = 0.5692
 seed  n_test_sites  n_test  test_outdoor_share
    0            15   16554            0.579256
    1            15   16513            0.565494
    2            15    7586            0.407725
    3            15   11282            0.528984
    4            15    9978            0.534676
mean test-fold majority (outdoor share) = 0.5232  <- the 0.560 of v1


## 1. E10a: v1 approximations (GBDT, sample re-weighting)

- Group-DRO approx.: weights from in-sample error rate per site
- JTT approx.: misclassified training rows up-weighted ×5

In [2]:
def group_dro_weights_approx(Xtr, ytr, gtr, seed):
    m0 = ev.make_model("gbdt", seed).fit(Xtr, ytr)
    err = pd.DataFrame({"g": gtr, "e": (m0.predict(Xtr) != ytr).astype(float)}).groupby("g").e.mean()
    w = pd.Series(gtr).map(err).to_numpy()
    return (w + 0.1) / (w + 0.1).mean()

def jtt_weights_approx(Xtr, ytr, seed, upweight=5.0):
    m0 = ev.make_model("gbdt", seed).fit(Xtr, ytr)
    w = np.ones(len(ytr)); w[m0.predict(Xtr) != ytr] = upweight
    return w

rows = []
for seed in SEEDS:
    a, b = site_split(seed, .3)
    if len(a) > CAP:
        a = np.random.RandomState(seed).choice(a, CAP, replace=False)
    r = ev.evaluate(Xoi[a], yoi[a], Xoi[b], yoi[b], "gbdt", seed)
    rows.append(dict(method="ERM (gbdt)", seed=seed, **{k: r[k] for k in ["acc", "bacc", "mf1", "base"]}))
    for nm, wf in [("GroupDRO_approx (gbdt, v1)", lambda: group_dro_weights_approx(Xoi[a], yoi[a], soi[a], seed)),
                   ("JTT_approx (gbdt, v1)", lambda: jtt_weights_approx(Xoi[a], yoi[a], seed))]:
        w = wf()
        p = ev.make_model("gbdt", seed).fit(Xoi[a], yoi[a], sample_weight=w).predict(Xoi[b])
        rows.append(dict(method=nm, seed=seed, acc=accuracy_score(yoi[b], p),
                         bacc=balanced_accuracy_score(yoi[b], p), mf1=f1_score(yoi[b], p, average="macro"),
                         base=pd.Series(yoi[b]).value_counts(normalize=True).max()))
e10a = pd.DataFrame(rows)

## 2. E10b: ERM, Group-DRO, JTT (MLP, `src/mitigation.py`)

- Backbone: MLP (1 hidden layer, 128 units), Adam, up to 20 epochs
- Groups: site × class (training sites)
- Validation: 20% of training sites; selection by worst-group accuracy (ERM: mean accuracy)
- Grid: ERM weight decay {1e-4, 1e-2}; Group-DRO η {0.01, 0.1} × weight decay {1e-4, 1e-2}; JTT T {1, 2} × λ {5, 20}
- Differences vs. ERM: bootstrap over seeds (paired splits)

In [3]:
try:
    from src import mitigation as mg
    rows_b = []
    for seed in SEEDS:
        a, b = site_split(seed, .3)
        if len(a) > CAP:
            a = np.sort(np.random.RandomState(seed).choice(a, CAP, replace=False))
        rows_b += mg.run_methods(Xoi, yoi, soi, a, b, seed=seed, quick=cfg.QUICK)
        print(f"  seed {seed} done", flush=True)
    e10b = pd.DataFrame(rows_b)
except ImportError as e:
    print("PyTorch not available - E10b skipped:", e)
    e10b = pd.DataFrame()

e10 = pd.concat([e10a, e10b], ignore_index=True)
e10.to_csv(cfg.RESULTS / "E10_mitigation_raw.csv", index=False)
s10 = e10.groupby("method").agg(acc=("acc", "mean"), sd=("acc", "std"), bacc=("bacc", "mean"),
                                mf1=("mf1", "mean"), base=("base", "mean"), n=("acc", "size")).reset_index()
if "worst_class_recall" in e10:
    s10 = s10.merge(e10.groupby("method").worst_class_recall.mean().reset_index(), on="method", how="left")
s10.to_csv(cfg.RESULTS / "E10_mitigation_summary.csv", index=False)
print(s10.round(4).to_string(index=False))
if len(e10b):
    print("\nselected hyper-parameters per seed:")
    print(e10b[["method", "seed", "params", "best_epoch", "val_score", "n_train", "n_val", "n_groups_train"]]
          .to_string(index=False))
    diffs = []
    erm = e10b[e10b.method == "ERM"].set_index("seed")
    for m in ["GroupDRO", "JTT"]:
        d = (e10b[e10b.method == m].set_index("seed")[["acc", "bacc"]] - erm[["acc", "bacc"]]).dropna()
        for mtr in ["acc", "bacc"]:
            ci = sfb.bootstrap_ci(d[mtr].to_numpy(), n_boot=cfg.N_BOOT)
            diffs.append(dict(method=m, metric=mtr, diff_vs_ERM=ci["mean"], lo=ci["lo"], hi=ci["hi"],
                              n_seeds=ci["n"], unit="seed (paired split)"))
    diffs = pd.DataFrame(diffs)
    diffs.to_csv(cfg.RESULTS / "E10_mitigation_diff_vs_ERM.csv", index=False)
    print(diffs.round(4).to_string(index=False))
for _, r in s10.iterrows():
    rp.record(f"D1.E10.{r.method}.acc", r.acc, "E10_mitigation_summary.csv", "R1-M9,R2-15",
              note=f"bacc={r.bacc:.4f}, base={r.base:.4f}, n_seeds={int(r.n)}")

  seed 0 done
  seed 1 done
  seed 2 done
  seed 3 done
  seed 4 done
                    method    acc     sd   bacc    mf1   base  n  worst_class_recall
                       ERM 0.5342 0.0166 0.5339 0.5252 0.5601  5              0.4240
                ERM (gbdt) 0.5016 0.0259 0.5019 0.4915 0.5601  5                 NaN
                  GroupDRO 0.5151 0.0119 0.5179 0.5134 0.5601  5              0.4803
GroupDRO_approx (gbdt, v1) 0.5061 0.0210 0.5064 0.4969 0.5601  5                 NaN
                       JTT 0.5086 0.0333 0.5082 0.4987 0.5601  5              0.4098
     JTT_approx (gbdt, v1) 0.5114 0.0224 0.5113 0.5031 0.5601  5                 NaN

selected hyper-parameters per seed:
  method  seed                                params  best_epoch  val_score  n_train  n_val  n_groups_train
     ERM     0                          {'wd': 0.01}           4   0.619028    15660   2449              36
GroupDRO     0              {'eta': 0.1, 'wd': 0.01}           9   0.448276    156

## 3. E11: conformal prediction sets

- Score: 1 − p̂(y | x); threshold: ⌈(n + 1)(1 − α)⌉-th smallest calibration score; Mondrian: per class
- α = 0.05, 0.10, 0.20
- Calibration designs: shared sites (calibration/test rows from the same held-out sites) / disjoint sites (different sites)

In [4]:
def e11_designs(seed):
    a, b = site_split(seed, .4)
    if len(a) > CAP:
        a = np.random.RandomState(seed).choice(a, CAP, replace=False)
    cal_s, te_s = train_test_split(b, test_size=.5, random_state=seed, stratify=yoi[b])
    gs = GroupShuffleSplit(1, test_size=.5, random_state=seed)
    c_rel, t_rel = next(gs.split(b, yoi[b], soi[b]))
    return a, {"shared_sites": (cal_s, te_s), "disjoint_sites": (b[c_rel], b[t_rel])}

rows, sel_rows, curves = [], [], []
for seed in SEEDS:
    a, designs = e11_designs(seed)
    model = ev.make_model("gbdt", seed).fit(Xoi[a], yoi[a])
    cls = model.classes_
    for dname, (cal, te) in designs.items():
        if len(np.unique(yoi[te])) < 2 or len(np.unique(yoi[cal])) < 2:
            continue
        for alpha in [0.05, 0.10, 0.20]:
            for mond in [False, True]:
                S_, info = ev.split_conformal(model, Xoi[cal], yoi[cal], Xoi[te], alpha=alpha,
                                              classes=cls, mondrian=mond)
                rows.append(dict(seed=seed, design=dname, alpha=alpha, mondrian=mond, target=1 - alpha,
                                 n_cal_sites=len(set(soi[cal])), n_test_sites=len(set(soi[te])),
                                 **ev.conformal_report(S_, yoi[te], cls),
                                 **{k: v for k, v in info.items()}))
        for alpha in [0.01, 0.05, 0.10, 0.20]:
            for weighting, g in [("record", None), ("site", soi[cal])]:
                for guarantee in ["expected", "high_prob"]:
                    r = ev.selective_risk_control(model, Xoi[cal], yoi[cal], Xoi[te], yoi[te], alpha=alpha,
                                                  groups_cal=g, guarantee=guarantee, delta=0.1)
                    r.update(seed=seed, design=dname)
                    sel_rows.append(r)
        if seed == 0:
            c = ev.risk_acceptance_curve(model, Xoi[te], yoi[te])
            c["design"] = dname
            curves.append(c)

e11 = pd.DataFrame(rows)
e11.to_csv(cfg.RESULTS / "E11_conformal_raw.csv", index=False)
s11 = e11.groupby(["design", "alpha", "mondrian"]).agg(
    coverage=("coverage", "mean"), cov_min=("coverage", "min"), target=("target", "mean"),
    set_size=("mean_set_size", "mean"), abstain=("abstain_rate", "mean"),
    referrals_per_1000=("referrals_per_1000", "mean"), n_decided=("n_decided", "mean"),
    acc_decided=("acc_on_decided", "mean"), base_decided=("baseline_on_decided", "mean"),
    n_cal=("n_cal", "mean"), n_test=("n_test", "mean"), n=("coverage", "size")).reset_index()
s11.to_csv(cfg.RESULTS / "E11_conformal_summary.csv", index=False)
print(s11.round(4).to_string(index=False))
for _, r in s11.iterrows():
    tag = f"D1.E11.{r.design}.alpha{r.alpha}.{'mondrian' if r.mondrian else 'split'}"
    rp.record(tag + ".coverage", r.coverage, "E11_conformal_summary.csv", "R2-12", note=f"target={r.target}")
    rp.record(tag + ".abstain", r.abstain, "E11_conformal_summary.csv", "R1(abstract),R2-12")
    rp.record(tag + ".acc_decided", r.acc_decided, "E11_conformal_summary.csv", "R1(abstract)",
              note=f"baseline on decided={r.base_decided:.4f}")

        design  alpha  mondrian  coverage  cov_min  target  set_size  abstain  referrals_per_1000  n_decided  acc_decided  base_decided  n_cal  n_test  n
disjoint_sites   0.05     False    0.9428   0.9150    0.95    1.8705   0.8705            870.5223     1680.6       0.5694        0.5424 5680.8 11884.2  5
disjoint_sites   0.05      True    0.9515   0.9203    0.95    1.8874   0.8874            887.3783     1440.0       0.5760        0.5436 5680.8 11884.2  5
disjoint_sites   0.10     False    0.8931   0.8437    0.90    1.7661   0.7661            766.0584     2968.4       0.5495        0.5626 5680.8 11884.2  5
disjoint_sites   0.10      True    0.9003   0.8639    0.90    1.7813   0.7813            781.2845     2752.4       0.5442        0.5533 5680.8 11884.2  5
disjoint_sites   0.20     False    0.7859   0.6756    0.80    1.5470   0.5470            546.9913     5674.6       0.5315        0.5683 5680.8 11884.2  5
disjoint_sites   0.20      True    0.7873   0.6981    0.80    1.5435   0.543

## 4. E11b: selective risk control

- α = 0.01, 0.05, 0.10, 0.20; δ = 0.1
- Weighting: record / site; guarantee: expected risk / high probability
- Joint risk = P(wrong and accepted); selective risk = P(wrong | accepted)
- `violations`: seeds with realised joint risk > α on the test rows (not a count of guarantee failures)
- Risk–acceptance curve: seed 0

In [ ]:
e11b = pd.DataFrame(sel_rows)
e11b.to_csv(cfg.RESULTS / "E11b_selective_risk_raw.csv", index=False)
s11b = e11b.groupby(["design", "weighting", "guarantee", "alpha"]).agg(
    threshold=("threshold", "mean"), n_cal_units=("n_cal_units", "mean"),
    n_test=("n_test", "mean"), n_accepted=("n_accepted", "mean"), accept_rate=("accept_rate", "mean"),
    referrals_per_1000=("referrals_per_1000", "mean"), joint_risk=("joint_risk", "mean"),
    violations=("risk_exceeds_alpha", "sum"), selective_risk=("selective_risk", "mean"),
    acc_accepted=("acc_on_accepted", "mean"), baseline_accepted=("baseline_on_accepted", "mean"),
    n=("threshold", "size")).reset_index()
s11b.to_csv(cfg.RESULTS / "E11b_selective_risk_summary.csv", index=False)
print(s11b.round(4).to_string(index=False))

# record vs. site weighting
print("\n--- record vs site weighting (shared sites, expected-risk guarantee) ---")
cmpw = s11b[(s11b.design == "shared_sites") & (s11b.guarantee == "expected")].pivot(
    index="alpha", columns="weighting", values=["selective_risk", "accept_rate", "n_accepted"])
print(cmpw.round(4).to_string())
for alpha, r in cmpw.iterrows():
    for w in ["record", "site"]:
        rp.record(f"D1.E11b.shared.expected.alpha{alpha}.{w}.selective_risk", r[("selective_risk", w)],
                  "E11b_selective_risk_summary.csv", "R1-M4,R2-13",
                  note=f"accept_rate={r[('accept_rate', w)]:.4f}, n_accepted={r[('n_accepted', w)]:.1f}")

curves = pd.concat(curves, ignore_index=True) if curves else pd.DataFrame()
curves.to_csv(cfg.RESULTS / "E11b_risk_acceptance_curve.csv", index=False)
if len(curves):
    best = curves[curves.accept_rate >= 0.10].selective_risk.min()
    print(f"\nlowest selective risk at >= 10 % acceptance (seed 0): {best:.4f}")
    rp.record("D1.E11b.curve.min_selective_risk_at_accept>=0.10", best, "E11b_risk_acceptance_curve.csv",
              "R1-M4,R2-13")

        design weighting guarantee  alpha  threshold  n_cal_units  n_test  n_accepted  accept_rate  referrals_per_1000  joint_risk  violations  selective_risk  acc_accepted  baseline_accepted  n
disjoint_sites    record  expected   0.01     0.9857       5680.8 11884.2       551.8       0.0401            959.8846      0.0155           3          0.3579        0.6421             0.6269  5
disjoint_sites    record  expected   0.05     0.9598       5680.8 11884.2      1674.2       0.1290            870.9922      0.0570           3          0.4306        0.5694             0.5420  5
disjoint_sites    record  expected   0.10     0.9246       5680.8 11884.2      2965.0       0.2337            766.2837      0.1068           3          0.4504        0.5496             0.5625  5
disjoint_sites    record  expected   0.20     0.8282       5680.8 11884.2      5672.2       0.4528            547.1607      0.2140           2          0.4685        0.5315             0.5683  5
disjoint_sites    record 